In [1]:
import os, sys
os.environ['HF_ENDPOINT'] = 'https://alpha.hf-mirror.com/'
os.environ["HF_HUB_URL"] = 'https://alpha.hf-mirror.com/'


In [ ]:
# from huggingface_hub import hf_hub_download
# hf_hub_download(repo_id="SpursgoZmy/MMTab", repo_type="dataset", filename="MMTab-eval_table_images_23K.zip")

'/root/.cache/huggingface/hub/datasets--SpursgoZmy--MMTab/snapshots/6335f59822b6f8a662d6d53cfcd6b71f8cf4fd62/MMTab-eval_table_images_23K.zip'

In [2]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from janus.models import MultiModalityCausalLM, VLChatProcessor
from janus.utils.io import load_pil_images

if torch.cuda.is_available():
    torch.device('cuda')
else:
    torch.device('cpu')

torch.cuda.empty_cache()

bnb_4bit_config = BitsAndBytesConfig(
    load_in_4bit=True,  
    bnb_4bit_compute_dtype=torch.float16,   
    bnb_4bit_use_double_quant=True,         
    bnb_4bit_quant_type="nf4",              # or "fp4"
)

# specify the path to the model
model_path = "deepseek-ai/Janus-Pro-1B"
checkpoint_path = "/root/autodl-tmp/Janus/ckpt-janus1b-4bit-output-1/checkpoint-4000"

vl_chat_processor: VLChatProcessor = VLChatProcessor.from_pretrained(
    model_path, 
    quantization_config=bnb_4bit_config,
    #  torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2"
    )
tokenizer = vl_chat_processor.tokenizer

vl_gpt: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(
    model_path, trust_remote_code=True
)


# vl_gpt: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(
#     checkpoint_path
# )

/root/miniconda3/envs/janus/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python version is above 3.10, patching the collections module.


/root/miniconda3/envs/janus/lib/python3.13/site-packages/transformers/models/auto/image_processing_auto.py:594: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughl

In [ ]:
def demo():
    vl_gpt = vl_gpt.to(torch.bfloat16).cuda().eval()
    demo_conversation = [
        {
            "role": "User",
            "content": "<image_placeholder>\nDescribe the content in the table",
            "images": ["/root/autodl-tmp/MMTab/pre_train_ds/images/table_pretrain_part_1/TABMWP_1.jpg"],
        },
        {"role": "Assistant", "content": ""},
    ]

    # load images and prepare for inputs
    pil_images = load_pil_images(demo_conversation)
    prepare_inputs = vl_chat_processor(
        conversations=demo_conversation, images=pil_images, force_batchify=True
    ).to(vl_gpt.device)
    # # run image encoder to get the image embeddings
    inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)
    print(inputs_embeds.shape)

    # # run the model to get the response
    outputs = vl_gpt.language_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=prepare_inputs.attention_mask,
        pad_token_id=tokenizer.eos_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=2048,
        do_sample=False,
        use_cache=True,
    )

    answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
    print(answer)
demo()

torch.Size([1, 630, 2048])
The table shows the number of plants per garden for different stem numbers. The stem numbers range from 3 to 8, and the leaf numbers are 33355, 6, 4578, 78, 2379, 689, and 689 again.


In [5]:
print(vl_gpt)

MultiModalityCausalLM(
  (vision_model): CLIPVisionTower(
    (vision_tower): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Identity()
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in

In [6]:
from datasets import load_dataset

pre_train_ds_path = "/root/autodl-tmp/MMTab/pre_train_ds/MMTab-pre_pretrain_data_llava_format_150K.json"
dataset = load_dataset("json", data_files=pre_train_ds_path, streaming=True)["train"]
shuffled_dataset = dataset.shuffle(seed=42, buffer_size=20)
print(shuffled_dataset)

IterableDataset({
    features: ['id', 'image', 'conversations'],
    num_shards: 1
})


In [7]:
def process_fn(row):
	images = row['image']
	conversations = row['conversations']
	row['janus_conversation'], row['answer'] = build_janus_template(images, conversations)
	return row


def build_janus_template(image, conversation):
	# convert one row of llava format to janus format
	llava_image_prompts = conversation[0]['value'].split('\n')
	prompt = llava_image_prompts[0] if llava_image_prompts[0].strip() != '<image>' else llava_image_prompts[1]
	answer = conversation[1]['value'].strip()
	return [
		{
			"role": "User",
			"content": f"<image_placeholder>\n{prompt}",
			"images": [f"/root/autodl-tmp/MMTab/pre_train_ds/images/{image}"],
		},
		{"role": "Assistant", "content": ""},
	], answer

# test_dataset = dataset.select(range(100)).map(process_fn, batched=False, remove_columns=dataset.column_names)
# print(test_dataset)
# print(test_dataset[0])
processed_dataset = shuffled_dataset.map(process_fn, batched=False, remove_columns=dataset.column_names)

In [8]:
def collate_fn(rows):
	batch_size = len(rows)
	if batch_size > 0:
		vl_chat_processor_outputs = []
		answers_ids = []
		for row in rows:
			# load images and prepare for inputs
			pil_images = load_pil_images(row["janus_conversation"])
			vl_chat_processor_output = vl_chat_processor(conversations=row["janus_conversation"], images=pil_images, force_batchify=False)
			
			answer_ids = tokenizer(row["answer"], add_special_tokens=False, return_tensors="pt").input_ids[0].flatten()
			# print(vl_chat_processor_output.input_ids)
			# print(answer_ids)
			input_ids = torch.cat([vl_chat_processor_output.input_ids, answer_ids, torch.tensor([tokenizer.eos_token_id])], dim=0)
			vl_chat_processor_output.sft_format = vl_chat_processor_output.sft_format + row["answer"]
			vl_chat_processor_output.input_ids = input_ids

			answers_ids.append(answer_ids)
			vl_chat_processor_outputs.append(vl_chat_processor_output)

		batched_prepare = vl_chat_processor.batchify(vl_chat_processor_outputs).to(vl_gpt.device)
		labels_tensor = torch.clone(batched_prepare.input_ids)
		
		for i in range(batch_size):
			answer_len_exclude_bos = answers_ids[i].shape[0] - 1
			labels_tensor[i, :-answer_len_exclude_bos] = -100
		
		# print("input:\n", batched_prepare.input_ids[0, -300:])
		# print("label:\n", labels_tensor[0, -300:])
		batch = {
			"input_ids":batched_prepare.input_ids,
			"attention_mask":batched_prepare.attention_mask,
			"pixel_values":batched_prepare.pixel_values,
			"images_seq_mask":batched_prepare.images_seq_mask,
			"images_emb_mask":batched_prepare.images_emb_mask,
			"labels": labels_tensor,
		}
	return batch

In [8]:
# import torch.nn as nn

# class VLMWrapper(nn.Module):
#     def __init__(self, vl_gpt):
#         super().__init__()
#         self.vl_gpt = vl_gpt

#     def forward(self, batch, input_ids=None, labels=None, **kwargs):
#         batched_prepare = batch["batched_prepare"]
#         inputs_embeds = vl_gpt.prepare_inputs_embeds(**batched_prepare)
#         labels=batch["labels"]

#         llm_outputs = self.vl_gpt.language_model(             
#             inputs_embeds=inputs_embeds,       
#             attention_mask=batched_prepare.attention_mask,      
#             labels=labels,                     
#             **kwargs
#         )
        
#         return llm_outputs  # includes .loss, .logits, etc.

# vlm_wrapper = VLMWrapper(vl_gpt)

In [ ]:
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training


lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],  # typical for LLaMA
)
# lora freeze all param 
vl_gpt = prepare_model_for_kbit_training(vl_gpt)

vl_gpt = get_peft_model(vl_gpt, lora_config)


for param in vl_gpt.aligner.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir="./ckpt-janus1b-4bit-output",
    max_steps=10000,
    per_device_train_batch_size=2,
    evaluation_strategy="no",
    # per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    # evaluation_strategy="epoch",
    save_strategy="steps",             
    save_steps=500,
    learning_rate=2e-4,
    bf16=True,  
    logging_steps=10,
    remove_unused_columns=False,
    dataloader_pin_memory=False
)

trainer = Trainer(
    model=vl_gpt,
    args=training_args,
    train_dataset=processed_dataset,
    data_collator=collate_fn,
)

trainer.train()

In [ ]:
trainable_params = 0
all_params = 0
for name, param in vl_gpt.named_parameters():
    all_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"Trainable: {name} => shape={param.shape}")
    else:
        print(f"Frozen: {name}")

print(f"Trainable = {trainable_params} / {all_params} params => {100 * trainable_params/all_params:.2f}%")


In [11]:
vl_gpt.print_trainable_parameters()

trainable params: 7,868,416 || all params: 2,090,804,875 || trainable%: 0.3763


In [7]:
import pandas as pd

eval_ds_path = "/root/autodl-tmp/MMTab/eval_ds/MMTab-eval_test_data_49K.json"

# Open and read the JSON file
df = pd.read_json(eval_ds_path)

tr_task_df = df[df["task_type"] == "TR"] 
tr_task_df.size
tr_task_df.iloc[0]["output"]

'<table border="1" cellspacing="0">\n<tr> <td rowspan="2"> year </td> <td rowspan="2"> team </td> <td colspan="2"> games </td> <td colspan="5"> receiving </td> <td colspan="5"> rushing </td> <td colspan="5"> returning </td> <td colspan="2"> fumbles </td> </tr>\n<tr> <td> gp </td> <td> gs </td> <td> rec </td> <td> yds </td> <td> avg </td> <td> lng </td> <td> td </td> <td> att </td> <td> yds </td> <td> avg </td> <td> lng </td> <td> td </td> <td> ret </td> <td> yds </td> <td> avg </td> <td> lng </td> <td> td </td> <td> fum </td> <td> lost </td> </tr>\n<tr> <td> 2012 </td> <td> dal </td> <td> 10 </td> <td> 0 </td> <td> 15 </td> <td> 128 </td> <td> 8.5 </td> <td> 20 </td> <td> 0 </td> <td> 0 </td> <td> 0 </td> <td> 0.0 </td> <td> 0 </td> <td> 0 </td> <td> 0 </td> <td> 0 </td> <td> 0.0 </td> <td> 0 </td> <td> 0 </td> <td> 0 </td> <td> 0 </td> </tr>\n<tr> <td> 2013 </td> <td> dal </td> <td> 14 </td> <td> 3 </td> <td> 39 </td> <td> 368 </td> <td> 9.4 </td> <td> 23 </td> <td> 2 </td> <td> 0 </t

In [9]:
def inference(user_prompt, image_id, model):
	sft_template = [
		{
			"role": "User",
			"content": f"<image_placeholder>\n{user_prompt}",
			"images": [f"/root/autodl-tmp/MMTab/eval_ds/all_test_image/{image_id}.jpg"],
		},
		{"role": "Assistant", "content": ""},
	]

	# load images and prepare for inputs
	pil_images = load_pil_images(sft_template)
	prepare_inputs = vl_chat_processor(
		conversations=sft_template, images=pil_images, force_batchify=True
	).to(model.device)
	# # run image encoder to get the image embeddings
	inputs_embeds = model.prepare_inputs_embeds(**prepare_inputs)

	# # run the model to get the response
	outputs = model.language_model.generate(
		inputs_embeds=inputs_embeds,
		attention_mask=prepare_inputs.attention_mask,
		pad_token_id=tokenizer.eos_token_id,
		bos_token_id=tokenizer.bos_token_id,
		eos_token_id=tokenizer.eos_token_id,
		max_new_tokens=8192,
		do_sample=False,
		use_cache=True,
	)

	answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
	return answer

In [10]:
from metrics import TEDS
import numpy as np
from tqdm import tqdm
teds = TEDS()
eval_size = 20


vl_gpt: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(
    model_path, trust_remote_code=True,
)


vl_gpt_ft: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
)

def run_eval(model):
	model = model.to(torch.bfloat16).cuda().eval()
	scores = []
	for i in tqdm(range(eval_size)):
		user_prompt = tr_task_df.iloc[i]["input"]
		image_id = tr_task_df.iloc[i]["image_id"]
		pred = inference(user_prompt, image_id, model)
		
		ground_truth = tr_task_df.iloc[i]["output"]
		score = teds.evaluate(pred, ground_truth)
		scores.append(score)
		print(score)
	
	return np.mean(scores)

print("original: ", run_eval(vl_gpt), "\nfine_tuned: ", run_eval(vl_gpt_ft))

	


  5%|▌         | 1/20 [00:58<18:35, 58.69s/it]

-0.03597122302158273


 10%|█         | 2/20 [01:10<09:17, 30.98s/it]

0.6868703851723186


 15%|█▌        | 3/20 [01:16<05:34, 19.66s/it]

...

0.23813327602948087


 95%|█████████▌| 19/20 [15:17<01:17, 77.23s/it]

-0.004541133792834051


100%|██████████| 20/20 [16:29<00:00, 49.48s/it]

-0.030501089324618702
original:  0.20388805876008273 
fine_tuned:  0.3781268352298951


In [5]:
pred = """<table border="1" cellspacing="0">
  <tr>
    <th colspan="2"><div>David Charles Hahn</div></th>
  </tr>
  <tr>
    <th>Born</th>
    <td>October 30, 1976<br/><div>Shelby Charter Township, Michigan, United States[ citation needed ]</div></td>
  </tr>
  <tr>
    <th>Died</th>
    <td>September 27, 2016 (aged 39)<br/><div>Shelby Charter Township, Michigan, United States</div></td>
  </tr>
  <tr>
    <th>Cause of death</th>
    <td>Alcohol poisoning</td>
  </tr>
  <tr>
    <th>Known for</th>
    <td>Building a nuclear reactor in his backyard at the age of 17</td>
  </tr>
</table>"""
teds.evaluate(pred, pred)

1.0

In [11]:
from lxml import html, etree

# Sample HTML
html_string = """
<table border="1" cellspacing="0">
  <tr>
    <th colspan="2"><div>David Charles Hahn</div></th>
  </tr>
  <tr>
    <th>Born</th>
    <td>October 30, 1976<br/><div>Shelby Charter Township, Michigan, United States[ citation needed ]</div></td>
  </tr>
  <tr>
    <th>Died</th>
    <td>September 27, 2016 (aged 39)<br/><div>Shelby Charter Township, Michigan, United States</div></td>
  </tr>
  <tr>
    <th>Cause of death</th>
    <td>Alcohol poisoning</td>
  </tr>
  <tr>
    <th>Known for</th>
    <td>Building a nuclear reactor in his backyard at the age of 17</td>
  </tr>
</table>
"""

# Parse the HTML
tree = html.fromstring(html_string)

# Pretty print the tree structure
print(etree.tostring(tree, pretty_print=True, encoding='unicode'))
print(tree.xpath('. | .//table'))

<table border="1" cellspacing="0">
  <tr>
    <th colspan="2"><div>David Charles Hahn</div></th>
  </tr>
  <tr>
    <th>Born</th>
    <td>October 30, 1976<br/><div>Shelby Charter Township, Michigan, United States[ citation needed ]</div></td>
  </tr>
  <tr>
    <th>Died</th>
    <td>September 27, 2016 (aged 39)<br/><div>Shelby Charter Township, Michigan, United States</div></td>
  </tr>
  <tr>
    <th>Cause of death</th>
    <td>Alcohol poisoning</td>
  </tr>
  <tr>
    <th>Known for</th>
    <td>Building a nuclear reactor in his backyard at the age of 17</td>
  </tr>
</table>


[<Element table at 0x7fbfce329950>]
